# Create Final Dataset

Create the full combined and balanced dataset from the source datasets

- **huggingface_hub** to download parquet shards locally
- **DuckDB** to sample from local parquet files
- **Polars** to clean, review, balance, and split the final dataset

## Here are some things we do in this notebook
1. Enumerate parquet files from the three datasets
2. Optionally keep only a random subset of shards for speed (Done specifically on the reddit dataset because it is especially large)
4. Sample large candidate pools with DuckDB (was able to compute random samples better than just using polars)
5. Clean, review, deduplicate, and cap dominance with Polars
6. Build a balanced final dataset
7. Create train / validation / test splits
8. Save all outputs to parquet so that they are cached and can be easily loaded in other files later

## Target data sizes for final dataset
- Candidate pools: **300,000 rows per platform** (used as a pool of clean data that we can then sample from for the final dataset)
- Final dataset: **300,000 total rows**
  - 100,000 Twitter
  - 100,000 Reddit
  - 100,000 Hacker News

We create a new fully balanced dataset sampling from the very large and unbalanced hugging face datasets.

In [1]:
%pip install duckdb polars pyarrow huggingface_hub scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import html
import random
import re

import duckdb
import polars as pl
from huggingface_hub import HfFileSystem, hf_hub_download
from sklearn.model_selection import train_test_split

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(14)

/Users/mhedlund/CIS2450/cis2450-final-project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


polars.config.Config

## Paths and configuration

Improtantly here we set a min and max text length for the post to get rid of any outliers that are extremely short or extremely long. This notebook is also the canonical dataset-building pipeline for the project: rerunning it end-to-end regenerates the cleaned candidate pools, final balanced dataset, and train/validation/test splits without relying on any external helper script.

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
IN_PROCESS_DIR = DATA_DIR / "in_process"
PROCESSED_DIR = DATA_DIR / "processed"

DOWNLOAD_DIR = IN_PROCESS_DIR / "hf_downloads"
CANDIDATE_DIR = IN_PROCESS_DIR / "candidate_pools_local_duckdb"
FINAL_DIR = PROCESSED_DIR / "final_dataset_local_duckdb"

for path in [DATA_DIR, IN_PROCESS_DIR, PROCESSED_DIR, DOWNLOAD_DIR, CANDIDATE_DIR, FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
rng = random.Random(SEED)

# Candidate-pool sizes
TWITTER_CANDIDATE_N = 300_000
REDDIT_CANDIDATE_N = 300_000
HN_CANDIDATE_N = 300_000

# Final per-platform target
FINAL_PER_PLATFORM_N = 100_000

# data split
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

# Some controls to try and filter data better, some not used (esp max_text_len not used for reddt)
MIN_TEXT_LEN = 20
MAX_TEXT_LEN = 500
MAX_AUTHOR_ROWS = 200
MAX_REDDIT_COMMUNITY_ROWS = 1_500

# fraction of datasets to use
TWITTER_SHARD_FRACTION = 0.1
REDDIT_SHARD_FRACTION = 1.0
HN_SHARD_FRACTION = 1.0

# Build and download toggles, toggle to True to fully redo these steps
DOWNLOAD_TWITTER = False
DOWNLOAD_REDDIT = False
DOWNLOAD_HN = False

BUILD_TWITTER = False
BUILD_REDDIT = False
BUILD_HN = False

print("Project root:", PROJECT_ROOT)
print("Download directory:", DOWNLOAD_DIR)
print("Candidate directory:", CANDIDATE_DIR)
print("Final directory:", FINAL_DIR)

Project root: /Users/mhedlund/CIS2450/cis2450-final-project
Download directory: /Users/mhedlund/CIS2450/cis2450-final-project/data/interim/hf_downloads
Candidate directory: /Users/mhedlund/CIS2450/cis2450-final-project/data/interim/candidate_pools_local_duckdb
Final directory: /Users/mhedlund/CIS2450/cis2450-final-project/data/processed/final_dataset_local_duckdb


## Dataset identifiers

In [4]:
TWITTER_DATASET = "enryu43/twitter100m_tweets"
REDDIT_DATASET = "anhchanghoangsg/reddit_pushshift_dataset_cleaned"
HN_DATASET = "open-index/hacker-news" 

## Hugging Face setup


## List parquet files from Hugging Face repos

In [ ]:
fs = HfFileSystem()

def list_files(repo_id: str) -> list[str]:
    root = f"datasets/{repo_id}"
    files = fs.find(root, detail=False)
    pre = root + "/"
    return sorted([f[len(pre):] if f.startswith(pre) else f for f in files])

twitter_repo_files = list_files(TWITTER_DATASET)
reddit_repo_files = list_files(REDDIT_DATASET)
hn_repo_files = list_files(HN_DATASET)

print("twitter files:", len(twitter_repo_files))
print("reddit files:", len(reddit_repo_files))
print("hn files:", len(hn_repo_files))

twitter repo files: 43
reddit repo files: 58382
hn repo files: 467


## Select parquet shards

In [ ]:
# Twitter: all parquet files
twitter_parquet_files = [
    f for f in twitter_repo_files
    if f.endswith(".parquet")
]

# Reddit: only submission files (these are posts only and not comments)
reddit_parquet_files = [
    f for f in reddit_repo_files
    if f.endswith(".parquet") and "submissions_cleaned" in f
]

# Hacker News: all parquet files; keep only type 2 comment-style content later in SQL
hn_parquet_files = [
    f for f in hn_repo_files
    if f.endswith(".parquet")
]

print("Twitter parquet files:", len(twitter_parquet_files))
print("Reddit submission parquet files:", len(reddit_parquet_files))
print("HN parquet files:", len(hn_parquet_files))

Twitter parquet files: 41
Reddit submission parquet files: 29227
HN parquet files: 463


## Choose subset of shards (to lower compute on Reddit)

In [ ]:
def choose_shards(files: list[str], fraction: float = 1.0, seed: int = 42) -> list[str]:
    files = sorted(files)
    if fraction >= 1.0:
        return files

    rng = random.Random(seed)
    chosen = files.copy()

    if fraction < 1.0:
        k = max(1, int(round(len(chosen) * fraction)))
        chosen = rng.sample(chosen, k=k)

    return sorted(chosen)

twitter_selected_files = choose_shards(
    twitter_parquet_files,
    fraction=TWITTER_SHARD_FRACTION,
    seed=SEED,
)

reddit_selected_files = choose_shards(
    reddit_parquet_files,
    fraction=REDDIT_SHARD_FRACTION,
    seed=SEED,
)

hn_selected_files = choose_shards(
    hn_parquet_files,
    fraction=HN_SHARD_FRACTION,
    seed=SEED,
)

print("Twitter selected shards:", len(twitter_selected_files))
print("Reddit selected shards:", len(reddit_selected_files))
print("HN selected shards:", len(hn_selected_files))

Twitter selected shards: 41
Reddit selected shards: 29227
HN selected shards: 463


## Download selected shards locally

This downloads only the selected shard paths and stores them in the data/in_process folder locally


In [ ]:
def download_repo_files(repo_id: str, repo_files: list[str], local_root: Path) -> list[str]:
    local_root.mkdir(parents=True, exist_ok=True)
    local_paths = []

    for i, repo_file in enumerate(repo_files, start=1):
        local_path = hf_hub_download(
            repo_id=repo_id,
            repo_type="dataset",
            filename=repo_file,
        )
        local_paths.append(local_path)

        # progress check
        if i % 25 == 0 or i == len(repo_files):
            print(f"{repo_id}: downloaded/resolved {i}/{len(repo_files)} files")

    return local_paths

In [9]:
TWITTER_LOCAL_LIST_PATH = DOWNLOAD_DIR / "twitter_local_files.txt"
REDDIT_LOCAL_LIST_PATH = DOWNLOAD_DIR / "reddit_local_files.txt"
HN_LOCAL_LIST_PATH = DOWNLOAD_DIR / "hn_local_files.txt"

if DOWNLOAD_TWITTER:
    twitter_local_paths = download_repo_files(TWITTER_DATASET, twitter_selected_files, DOWNLOAD_DIR / "twitter")
    TWITTER_LOCAL_LIST_PATH.write_text("\n".join(twitter_local_paths))
else:
    twitter_local_paths = TWITTER_LOCAL_LIST_PATH.read_text().splitlines()

if DOWNLOAD_REDDIT:
    reddit_local_paths = download_repo_files(REDDIT_DATASET, reddit_selected_files, DOWNLOAD_DIR / "reddit")
    REDDIT_LOCAL_LIST_PATH.write_text("\n".join(reddit_local_paths))
else:
    reddit_local_paths = REDDIT_LOCAL_LIST_PATH.read_text().splitlines()

if DOWNLOAD_HN:
    hn_local_paths = download_repo_files(HN_DATASET, hn_selected_files, DOWNLOAD_DIR / "hn")
    HN_LOCAL_LIST_PATH.write_text("\n".join(hn_local_paths))
else:
    hn_local_paths = HN_LOCAL_LIST_PATH.read_text().splitlines()

print("Local Twitter files:", len(twitter_local_paths))
print("Local Reddit files:", len(reddit_local_paths))
print("Local HN files:", len(hn_local_paths))

Local Twitter files: 41
Local Reddit files: 2923
Local HN files: 463


## Make DuckDB connection

In [10]:
con = duckdb.connect(database=str(CANDIDATE_DIR / "sampling_local.duckdb"))
con.execute("PRAGMA enable_progress_bar;")
con.execute("SET threads TO 4;")

## DuckDB Sampling Helpers

In [ ]:
def sql_list(paths: list[str]) -> str:
    return "[" + ", ".join("'" + p.replace("'", "''") + "'" for p in paths) + "]"

def duckdb_sample_to_parquet(query_sql: str, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    path_out = str(out_path).replace("'", "''")
    copy_sql = f"COPY ({query_sql}) TO '{path_out}' (FORMAT PARQUET);"
    con.execute(copy_sql)
    print(f"Saved sample to {out_path}")

## Define DuckDB sampling queries

We use DuckDB to take our samples from the datasets and standardize overlapping fields like platform, post_id, author, text, and created_at, while keeping some domain specific fields like retweets. For Hacker News we intentionally keep only `type = 2` rows, since those comment-style posts are the closest match to the short, user-authored conversational text we want to compare against Reddit submissions and tweets.

In [ ]:
twitter_query = f'''
SELECT
    'twitter' AS platform,
    CAST(id AS VARCHAR) AS post_id,
    CAST("user" AS VARCHAR) AS author,
    CAST(tweet AS VARCHAR) AS text,
    TRY_CAST(date AS TIMESTAMP) AS created_at,
    TRY_CAST(likes AS BIGINT) AS likes,
    TRY_CAST(retweets AS BIGINT) AS retweets,
    TRY_CAST(replies AS BIGINT) AS replies,
    TRY_CAST(quotes AS BIGINT) AS quotes
FROM read_parquet({sql_list(twitter_local_paths)}, union_by_name=true)
WHERE tweet IS NOT NULL
  AND length(trim(CAST(tweet AS VARCHAR))) >= {MIN_TEXT_LEN}
USING SAMPLE reservoir({TWITTER_CANDIDATE_N} ROWS)
'''

# time stuff here was a bit weird to deal with, as well as properly getting some fields cause dataset was a bit off
reddit_query = f'''
SELECT
    'reddit' AS platform,
    CAST(name AS VARCHAR) AS post_id,
    CAST(author AS VARCHAR) AS author,
    trim(
        concat(
            coalesce(CAST(title AS VARCHAR), ''),
            ' ',
            coalesce(CAST(selftext AS VARCHAR), '')
        )
    ) AS text,
    COALESCE(
    CAST(to_timestamp(TRY_CAST(created_utc AS BIGINT)) AS TIMESTAMP),
    TRY_CAST(created_utc AS TIMESTAMP)
    ) AS created_at,
    TRY_CAST(score AS BIGINT) AS score,
    CAST(subreddit AS VARCHAR) AS community,
    TRY_CAST(num_comments AS BIGINT) AS num_comments,
    TRY_CAST(upvote_ratio AS DOUBLE) AS upvote_ratio
FROM read_parquet({sql_list(reddit_local_paths)}, union_by_name=true)
WHERE (title IS NOT NULL OR selftext IS NOT NULL)
  AND trim(
        concat(
            coalesce(CAST(title AS VARCHAR), ''),
            ' ',
            coalesce(CAST(selftext AS VARCHAR), '')
        )
      ) NOT IN ('[deleted]', '[removed]')
  AND length(
        trim(
            concat(
                coalesce(CAST(title AS VARCHAR), ''),
                ' ',
                coalesce(CAST(selftext AS VARCHAR), '')
            )
        )
      ) >= {MIN_TEXT_LEN}
USING SAMPLE reservoir({REDDIT_CANDIDATE_N} ROWS)
'''

hn_query = f'''
SELECT
    'hackernews' AS platform,
    CAST(id AS VARCHAR) AS post_id,
    CAST("by" AS VARCHAR) AS author,
    CAST(text AS VARCHAR) AS text,
    TRY_CAST(time AS TIMESTAMP) AS created_at,
    TRY_CAST(score AS BIGINT) AS score,
    CAST(parent AS VARCHAR) AS parent_id
FROM read_parquet({sql_list(hn_local_paths)}, union_by_name=true)
WHERE type = 2
  AND text IS NOT NULL
  AND length(trim(CAST(text AS VARCHAR))) >= {MIN_TEXT_LEN}
USING SAMPLE reservoir({HN_CANDIDATE_N} ROWS)
''' 

## Candidate-pool output paths

In [13]:
TWITTER_CANDIDATE_PATH = CANDIDATE_DIR / f"twitter_candidates_{TWITTER_CANDIDATE_N}.parquet"
REDDIT_CANDIDATE_PATH = CANDIDATE_DIR / f"reddit_candidates_{REDDIT_CANDIDATE_N}.parquet"
HN_CANDIDATE_PATH = CANDIDATE_DIR / f"hackernews_candidates_{HN_CANDIDATE_N}.parquet" 

## Run DuckDB sampling

In [14]:
if BUILD_TWITTER:
    duckdb_sample_to_parquet(twitter_query, TWITTER_CANDIDATE_PATH)
else:
    print("Skipping Twitter build")

if BUILD_REDDIT:
    duckdb_sample_to_parquet(reddit_query, REDDIT_CANDIDATE_PATH)
else:
    print("Skipping Reddit build")

if BUILD_HN:
    duckdb_sample_to_parquet(hn_query, HN_CANDIDATE_PATH)
else:
    print("Skipping Hacker News build")

100% ▕██████████████████████████████████████▏ (00:00:16.83 elapsed)     
Saved sample to /Users/mhedlund/CIS2450/cis2450-final-project/data/interim/candidate_pools_local_duckdb/twitter_candidates_300000.parquet
100% ▕██████████████████████████████████████▏ (00:00:13.93 elapsed)     
Saved sample to /Users/mhedlund/CIS2450/cis2450-final-project/data/interim/candidate_pools_local_duckdb/reddit_candidates_300000.parquet
100% ▕██████████████████████████████████████▏ (00:00:08.46 elapsed)     
Saved sample to /Users/mhedlund/CIS2450/cis2450-final-project/data/interim/candidate_pools_local_duckdb/hackernews_candidates_300000.parquet


## Use Polars to continue cleaning

In [15]:
twitter_raw = pl.read_parquet(TWITTER_CANDIDATE_PATH)
reddit_raw = pl.read_parquet(REDDIT_CANDIDATE_PATH)
hn_raw = pl.read_parquet(HN_CANDIDATE_PATH)

print(twitter_raw.shape, reddit_raw.shape, hn_raw.shape)

(288204, 9) (280951, 9) (248190, 7)


## Cleaning and modification helpers

Before modeling, we apply a small set of preprocessing rules that are meant to reduce obvious source bias instead of maximizing raw row count. We keep only non-null text, author, and timestamp fields; remove deleted or empty text; strip HTML-like markup artifacts from the text; enforce a minimum text length of 20 characters; deduplicate repeated text within each platform; cap overrepresented authors on every platform and overrepresented Reddit communities; and later balance the final dataset across both platform and time buckets. We also avoid training on metadata like author, post id, or timestamp so the classifier has to learn from writing style rather than platform-specific record structure.

In [ ]:
HTML_TAG_RE = re.compile(r"<[^>]+>")

def basic_text_clean(text: str | None) -> str | None:
    if text is None:
        return None
    text = html.unescape(str(text))
    text = HTML_TAG_RE.sub(" ", text)
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    if text == "":
        return None
    return text

def add_time_features(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("created_at").dt.year().alias("year"),
        pl.col("created_at").dt.month().alias("month"),
        pl.col("created_at").dt.strftime("%Y-%m").alias("year_month"),
    ])

def clean_pool(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.with_columns([
            pl.col("text").map_elements(basic_text_clean, return_dtype=pl.Utf8).alias("text"),
            pl.col("author").cast(pl.Utf8, strict=False).alias("author"),
        ])
        .filter(pl.col("text").is_not_null())
        .filter(pl.col("author").is_not_null())
        .filter(pl.col("created_at").is_not_null())
        .filter(~pl.col("text").is_in(["[deleted]", "[removed]"]))
        .with_columns(pl.col("text").str.len_chars().alias("text_len"))
        .filter(pl.col("text_len") >= MIN_TEXT_LEN)
        # .filter(pl.col("text_len") <= MAX_TEXT_LEN)
        .pipe(add_time_features)
    )

def dedupe_text_within_platform(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.with_columns(
            pl.col("text")
            .str.to_lowercase()
            .str.replace_all(r"\s+", " ")
            .str.strip_chars()
            .alias("text_norm")
        )
        .unique(subset=["platform", "text_norm"], keep="first")
        .drop("text_norm")
    )

def cap_group_size(df: pl.DataFrame, col: str, max_per_group: int, seed: int = 42) -> pl.DataFrame:
    if col not in df.columns or df.height == 0:
        return df
    return df.group_by(col).map_groups(
        lambda g: g.sample(n=min(len(g), max_per_group), shuffle=True, seed=seed)
    )

def review_pool(df: pl.DataFrame, name: str, group_col: str | None = None) -> None:
    print(f"===== AUDIT: {name} =====")
    print(df.select([
        pl.len().alias("n_rows"),
        pl.col("author").n_unique().alias("unique_authors"),
        pl.col("text").n_unique().alias("unique_texts"),
        pl.col("year_month").n_unique().alias("unique_year_months"),
        pl.col("text_len").mean().alias("mean_text_len"),
        pl.col("text_len").median().alias("median_text_len"),
    ]))
    print("\nTop authors")
    print(df.group_by("author").len().sort("len", descending=True).head(10))
    if group_col and group_col in df.columns:
        print(f"\nTop {group_col}")
        print(df.group_by(group_col).len().sort("len", descending=True).head(10))
    print("\nTop months")
    print(df.group_by("year_month").len().sort("len", descending=True).head(10))
    print()

# trying to balance samples over time buckets to avoid bad behavior
def time_balanced_sample(df: pl.DataFrame, target_n: int, bucket_col: str = "year_month", seed: int = 42) -> pl.DataFrame:
    if df.height < target_n:
        raise ValueError(f"Requested {target_n:,} rows but only have {df.height:,} rows.")

    counts = (
        df.group_by(bucket_col)
        .len()
        .rename({"len": "bucket_count"})
        .sort(bucket_col)
    )

    total = counts["bucket_count"].sum()
    allocations = []
    remaining = target_n

    for row in counts.iter_rows(named=True):
        bucket = row[bucket_col]
        bucket_count = row["bucket_count"]
        take_n = min(bucket_count, int(round(target_n * bucket_count / total)))
        allocations.append({"bucket": bucket, "bucket_count": bucket_count, "take_n": take_n})
        remaining -= take_n

    if remaining > 0:
        allocations = sorted(allocations, key=lambda x: (x["bucket_count"] - x["take_n"]), reverse=True)
        i = 0
        while remaining > 0 and allocations:
            room = allocations[i]["bucket_count"] - allocations[i]["take_n"]
            if room > 0:
                allocations[i]["take_n"] += 1
                remaining -= 1
            i = (i + 1) % len(allocations)
    elif remaining < 0:
        extra = -remaining
        allocations = sorted(allocations, key=lambda x: x["take_n"], reverse=True)
        i = 0
        while extra > 0 and allocations:
            if allocations[i]["take_n"] > 0:
                allocations[i]["take_n"] -= 1
                extra -= 1
            i = (i + 1) % len(allocations)

    parts = []
    for alloc in allocations:
        if alloc["take_n"] <= 0:
            continue
        bucket_df = df.filter(pl.col(bucket_col) == alloc["bucket"])
        parts.append(bucket_df.sample(n=alloc["take_n"], shuffle=True, seed=seed))

    out = pl.concat(parts, how="vertical") if parts else pl.DataFrame()
    if out.height > target_n:
        out = out.sample(n=target_n, shuffle=True, seed=seed)
    elif out.height < target_n:
        deficit = target_n - out.height
        used_ids = set(out["post_id"].to_list())
        remainder = df.filter(~pl.col("post_id").is_in(list(used_ids)))
        out = pl.concat([out, remainder.sample(n=deficit, shuffle=True, seed=seed)], how="vertical")

    return out.sample(fraction=1.0, shuffle=True, seed=seed)

def split_train_val_test(df: pl.DataFrame, seed: int = 42):
    pdf = df.to_pandas()
    train_pdf, temp_pdf = train_test_split(
        pdf,
        test_size=(1 - TRAIN_FRAC),
        random_state=seed,
        stratify=pdf["platform"],
    )
    val_relative = VAL_FRAC / (VAL_FRAC + TEST_FRAC)
    val_pdf, test_pdf = train_test_split(
        temp_pdf,
        train_size=val_relative,
        random_state=seed,
        stratify=temp_pdf["platform"],
    )
    return pl.from_pandas(train_pdf), pl.from_pandas(val_pdf), pl.from_pandas(test_pdf)

## Clean candidate pools

In [17]:
twitter_clean = clean_pool(twitter_raw)
reddit_clean = clean_pool(reddit_raw)
hn_clean = clean_pool(hn_raw)

twitter_clean = dedupe_text_within_platform(twitter_clean)
reddit_clean = dedupe_text_within_platform(reddit_clean)
hn_clean = dedupe_text_within_platform(hn_clean)

twitter_clean = cap_group_size(twitter_clean, "author", MAX_AUTHOR_ROWS, seed=SEED)

reddit_clean = cap_group_size(reddit_clean, "author", MAX_AUTHOR_ROWS, seed=SEED)
reddit_clean = cap_group_size(reddit_clean, "community", MAX_REDDIT_COMMUNITY_ROWS, seed=SEED)

hn_clean = cap_group_size(hn_clean, "author", MAX_AUTHOR_ROWS, seed=SEED)

print(twitter_clean.shape, reddit_clean.shape, hn_clean.shape)

(287925, 13) (255545, 13) (246885, 11)


## Review candidate pools

At this point the cleaned candidate pools reflect the core dataset assumptions we want to defend in the final report: short conversational text only, no null or deleted rows, deduplicated text within each platform, capped dominance from prolific authors or giant communities, and no metadata features that would let the models cheat by learning platform identity from source-specific fields.

## Audit remaining HTML-like content

Because Hacker News text often contains markup from the source dataset, we explicitly audit for any remaining HTML-like patterns after `basic_text_clean(...)` runs. This helps us show that the cleaning step is actually removing platform-specific markup artifacts rather than just assuming it does.

In [ ]:
HTML_AUDIT_PATTERNS = {
    "contains_any_tag": r"<[^>]+>",
    "contains_p_tag": r"<p>",
    "contains_href": r"href=",
    "contains_nofollow": r"nofollow",
}

def audit_remaining_html_like_content(df: pl.DataFrame, label: str) -> pl.DataFrame:
    total_rows = df.height
    rows = []
    for pattern_name, pattern in HTML_AUDIT_PATTERNS.items():
        match_count = df.filter(pl.col("text").str.contains(pattern)).height
        rows.append({
            "platform": label,
            "pattern": pattern_name,
            "matches": match_count,
            "pct_of_rows": round((match_count / total_rows) * 100, 4) if total_rows else 0.0,
        })
    return pl.DataFrame(rows)

html_audit = pl.concat([
    audit_remaining_html_like_content(twitter_clean, "twitter"),
    audit_remaining_html_like_content(reddit_clean, "reddit"),
    audit_remaining_html_like_content(hn_clean, "hackernews"),
], how="vertical")

html_audit.sort(["platform", "pattern"])


In [ ]:
time_balanced_sample(twitter_clean, "Twitter")
time_balanced_sample(reddit_clean, "Reddit", group_col="community")
time_balanced_sample(hn_clean, "Hacker News")

===== AUDIT: Twitter =====
shape: (1, 6)
┌────────┬────────────────┬──────────────┬────────────────────┬───────────────┬─────────────────┐
│ n_rows ┆ unique_authors ┆ unique_texts ┆ unique_year_months ┆ mean_text_len ┆ median_text_len │
│ ---    ┆ ---            ┆ ---          ┆ ---                ┆ ---           ┆ ---             │
│ u32    ┆ u32            ┆ u32          ┆ u32                ┆ f64           ┆ f64             │
╞════════╪════════════════╪══════════════╪════════════════════╪═══════════════╪═════════════════╡
│ 287925 ┆ 96262          ┆ 287925       ┆ 188                ┆ 123.714815    ┆ 107.0           │
└────────┴────────────────┴──────────────┴────────────────────┴───────────────┴─────────────────┘

Top authors
shape: (10, 2)
┌────────────────┬─────┐
│ author         ┆ len │
│ ---            ┆ --- │
│ str            ┆ u32 │
╞════════════════╪═════╡
│ alebrugues     ┆ 13  │
│ PhotosByCollis ┆ 13  │
│ kencen         ┆ 13  │
│ Tatsuya_MOD    ┆ 12  │
│ StJoHospice    ┆ 1

## Save cleaned candidate pools

In [19]:
TWITTER_CLEAN_CANDIDATE_PATH = CANDIDATE_DIR / f"twitter_candidates_clean_{twitter_clean.height}.parquet"
REDDIT_CLEAN_CANDIDATE_PATH = CANDIDATE_DIR / f"reddit_candidates_clean_{reddit_clean.height}.parquet"
HN_CLEAN_CANDIDATE_PATH = CANDIDATE_DIR / f"hackernews_candidates_clean_{hn_clean.height}.parquet"

twitter_clean.write_parquet(TWITTER_CLEAN_CANDIDATE_PATH)
reddit_clean.write_parquet(REDDIT_CLEAN_CANDIDATE_PATH)
hn_clean.write_parquet(HN_CLEAN_CANDIDATE_PATH)

print(TWITTER_CLEAN_CANDIDATE_PATH)
print(REDDIT_CLEAN_CANDIDATE_PATH)
print(HN_CLEAN_CANDIDATE_PATH)

/Users/mhedlund/CIS2450/cis2450-final-project/data/interim/candidate_pools_local_duckdb/twitter_candidates_clean_287925.parquet
/Users/mhedlund/CIS2450/cis2450-final-project/data/interim/candidate_pools_local_duckdb/reddit_candidates_clean_255545.parquet
/Users/mhedlund/CIS2450/cis2450-final-project/data/interim/candidate_pools_local_duckdb/hackernews_candidates_clean_246885.parquet


## Build the balanced base dataset

We combine the datasets from each source.

In [ ]:
twitter_final = time_balanced_sample(twitter_clean, FINAL_PER_PLATFORM_N, seed=SEED)
reddit_final = time_balanced_sample(reddit_clean, FINAL_PER_PLATFORM_N, seed=SEED)
hn_final = time_balanced_sample(hn_clean, FINAL_PER_PLATFORM_N, seed=SEED)

final_dataset = (
    pl.concat([twitter_final, reddit_final, hn_final], how="diagonal_relaxed")
    .sample(fraction=1.0, shuffle=True, seed=SEED)
)

final_dataset.shape

(300000, 18)

## Collected dataset audits

In [21]:
print(final_dataset.group_by("platform").len().sort("platform"))

print(final_dataset.group_by("platform").agg([
    pl.col("text_len").mean().alias("mean_text_len"),
    pl.col("text_len").median().alias("median_text_len"),
    pl.col("year").min().alias("min_year"),
    pl.col("year").max().alias("max_year"),
]).sort("platform"))

print(final_dataset.group_by(["platform", "year_month"]).len().sort(["platform", "year_month"]).tail(30))

shape: (3, 2)
┌────────────┬────────┐
│ platform   ┆ len    │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ hackernews ┆ 100000 │
│ reddit     ┆ 100000 │
│ twitter    ┆ 100000 │
└────────────┴────────┘
shape: (3, 5)
┌────────────┬───────────────┬─────────────────┬──────────┬──────────┐
│ platform   ┆ mean_text_len ┆ median_text_len ┆ min_year ┆ max_year │
│ ---        ┆ ---           ┆ ---             ┆ ---      ┆ ---      │
│ str        ┆ f64           ┆ f64             ┆ i32      ┆ i32      │
╞════════════╪═══════════════╪═════════════════╪══════════╪══════════╡
│ hackernews ┆ 375.66357     ┆ 246.0           ┆ 2007     ┆ 2026     │
│ reddit     ┆ 307.22964     ┆ 90.0            ┆ 2011     ┆ 2024     │
│ twitter    ┆ 123.67065     ┆ 107.0           ┆ 2007     ┆ 2023     │
└────────────┴───────────────┴─────────────────┴──────────┴──────────┘
shape: (30, 3)
┌──────────┬────────────┬──────┐
│ platform ┆ year_month ┆ len  │
│ ---      ┆ ---        ┆ ---  │
│ s

## Save the full dataset

In [22]:
FINAL_DATASET_PATH = FINAL_DIR / "social_platform_dataset_300k.parquet"
final_dataset.write_parquet(FINAL_DATASET_PATH)
print("Saved:", FINAL_DATASET_PATH)

Saved: /Users/mhedlund/CIS2450/cis2450-final-project/data/processed/final_dataset_local_duckdb/social_platform_dataset_300k.parquet


## Train, validation, test split

In [23]:
train_df, val_df, test_df = split_train_val_test(final_dataset, seed=SEED)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\nTrain platform balance")
print(train_df.group_by("platform").len().sort("platform"))

print("\nValidation platform balance")
print(val_df.group_by("platform").len().sort("platform"))

print("\nTest platform balance")
print(test_df.group_by("platform").len().sort("platform"))

train: (209999, 18)
val: (45000, 18)
test: (45001, 18)

Train platform balance
shape: (3, 2)
┌────────────┬───────┐
│ platform   ┆ len   │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ hackernews ┆ 70000 │
│ reddit     ┆ 70000 │
│ twitter    ┆ 69999 │
└────────────┴───────┘

Validation platform balance
shape: (3, 2)
┌────────────┬───────┐
│ platform   ┆ len   │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ hackernews ┆ 15000 │
│ reddit     ┆ 15000 │
│ twitter    ┆ 15000 │
└────────────┴───────┘

Test platform balance
shape: (3, 2)
┌────────────┬───────┐
│ platform   ┆ len   │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ hackernews ┆ 15000 │
│ reddit     ┆ 15000 │
│ twitter    ┆ 15001 │
└────────────┴───────┘


In [ ]:
# investifate the time balance of the data to make sure there are no extreme outliers
twitter_clean.group_by("year").len().sort("year")
reddit_clean.group_by("year").len().sort("year")
hn_clean.group_by("year").len().sort("year")

year,len
i32,u32
2007,463
2008,1480
2009,3022
2010,5078
2011,6189
2012,7562
…,…
2021,22460
2022,24135


The balance of the various sources doesn't seem too extreme. We check this mainly to make sure that we aren't capturing linguistic differences across time rather than across platform. In this case even though the platforms span across different dates, because they are generally all quite similar we don't think it will affect the style of the posts too much. Additioanlly the models themselves won't be trained with the date of the post as a feature so they can't just learn which platforms have data from which dates.

## Save split datasets

In [ ]:
TRAIN_PATH = FINAL_DIR / "train_210k.parquet"
VAL_PATH = FINAL_DIR / "val_45k.parquet"
TEST_PATH = FINAL_DIR / "test_45k.parquet"

train_df.write_parquet(TRAIN_PATH)
val_df.write_parquet(VAL_PATH)
test_df.write_parquet(TEST_PATH)

print(TRAIN_PATH)
print(VAL_PATH)
print(TEST_PATH)

/Users/mhedlund/CIS2450/cis2450-final-project/data/processed/final_dataset_local_duckdb/train_210k.parquet
/Users/mhedlund/CIS2450/cis2450-final-project/data/processed/final_dataset_local_duckdb/val_45k.parquet
/Users/mhedlund/CIS2450/cis2450-final-project/data/processed/final_dataset_local_duckdb/test_45k.parquet
